In [ ]:
"""
================================================================================
AHNN — Full Model Comparison (AHNN vs 10 Competitors)
================================================================================
Thesis-aligned comparison jaisa roadmap PDF mein likha hai.

Models compared:
  1. AHNN         (your proposed model)
  2. LSTM         (recurrent baseline)
  3. GRU          (recurrent baseline)
  4. RNN          (vanilla recurrent)
  5. XGBoost
  6. LightGBM
  7. RandomForest
  8. GBM
  9. SVM-RBF
 10. MLP
 11. LogReg       (simple linear baseline)

Metrics reported per model:
  Accuracy | F1 | AUC | PR-AUC | Brier | Hamming | Euclidean |
  Lyapunov | FairnessGap | PredVol (bootstrap variance)

Graphs generated:
  Fig 11 — AHNN vs All: multi-metric radar + bar comparison
  Fig 12 — Composite Ranking (Equal + Regulator-aligned Basel III)
  Fig 13 — Friedman Test (AHNN vs competitors across folds)
  Fig 14 — KS Test per feature
  Fig 15 — Prediction Variance across bootstrap (AHNN)

Run:  python ahnn_comparison.py
================================================================================
"""

from __future__ import annotations

import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.base import clone, BaseEstimator, ClassifierMixin
from sklearn.metrics import (
    roc_auc_score, f1_score, accuracy_score,
    brier_score_loss, average_precision_score,
)

try:
    from xgboost import XGBClassifier
    HAS_XGB = True
except ImportError:
    HAS_XGB = False

try:
    from lightgbm import LGBMClassifier
    HAS_LGBM = True
except ImportError:
    HAS_LGBM = False

warnings.filterwarnings("ignore")

# Aapke existing file se sab import
import ahnn_all_in_one
import importlib
importlib.reload(ahnn_all_in_one)
from ahnn_all_in_one import *

os.makedirs(OUT_DIR, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 150,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.3,
})


# ==============================================================================
# AHNN IMPLEMENTATION (pure numpy — matches PDF spec)
# ------------------------------------------------------------------------------
#   p(X) = Σ_t α_t · σ(x_t^T β_t + b_t)
#   α_t = softmax(γ_t)
#   Constraint: ||β_t|| ≤ ρ  (spectral clipping → Lyapunov stability)
#   Loss: BCE + λ_fair · (µ_small - µ_large)^2 + λ_wd · Σ||β||²
# ==============================================================================

class AHNNModel(BaseEstimator, ClassifierMixin):
    """
    Adaptive Hybrid Neural Network — numpy implementation.
    Input: X of shape [N, T, F]
    Output: probability in [0, 1]
    """

    def __init__(self,
                 T: int = T_YEARS,
                 F: int = F_FEATURES,
                 lr: float = LEARNING_RATE,
                 epochs: int = EPOCHS,
                 rho: float = SPECTRAL_BOUND_RHO,
                 lam_fair: float = FAIRNESS_LAMBDA,
                 lam_wd: float = WEIGHT_DECAY,
                 dropout_p: float = DROPOUT_P,
                 random_state: int = RANDOM_SEED):
        self.T = T
        self.F = F
        self.lr = lr
        self.epochs = epochs
        self.rho = rho
        self.lam_fair = lam_fair
        self.lam_wd = lam_wd
        self.dropout_p = dropout_p
        self.random_state = random_state

    # ---------- internal helpers ----------
    @staticmethod
    def _sigmoid(z):
        return 1.0 / (1.0 + np.exp(-np.clip(z, -30, 30)))

    @staticmethod
    def _softmax(logits):
        e = np.exp(logits - logits.max())
        return e / e.sum()

    def _init_params(self):
        rng = np.random.default_rng(self.random_state)
        self.beta_  = 0.1 * rng.standard_normal((self.T, self.F))   # [T, F]
        self.b_     = np.zeros(self.T)                              # [T]
        self.gamma_ = np.zeros(self.T)                              # gate logits

        # Adam state
        self._m = {k: np.zeros_like(v) for k, v in
                   [("beta", self.beta_), ("b", self.b_), ("gamma", self.gamma_)]}
        self._v = {k: np.zeros_like(v) for k, v in
                   [("beta", self.beta_), ("b", self.b_), ("gamma", self.gamma_)]}
        self._step = 0

    def _forward(self, X):
        # X: [N, T, F] → returns p:[N], pt:[N,T], alpha:[T]
        N = X.shape[0]
        # z_t = x_t · β_t + b_t  → shape [N, T]
        z = np.einsum("ntf,tf->nt", X, self.beta_) + self.b_[None, :]
        pt = self._sigmoid(z)                               # [N, T]
        alpha = self._softmax(self.gamma_)                  # [T]
        p = pt @ alpha                                       # [N]
        return p, pt, alpha

    def _adam_update(self, name, param, grad,
                     beta1=0.9, beta2=0.999, eps=1e-8):
        self._m[name] = beta1 * self._m[name] + (1 - beta1) * grad
        self._v[name] = beta2 * self._v[name] + (1 - beta2) * grad * grad
        m_hat = self._m[name] / (1 - beta1 ** self._step)
        v_hat = self._v[name] / (1 - beta2 ** self._step)
        return param - self.lr * m_hat / (np.sqrt(v_hat) + eps)

    def _spectral_clip(self):
        # ||β_t|| ≤ ρ for all t
        norms = np.linalg.norm(self.beta_, axis=1)            # [T]
        mask = norms > self.rho
        if mask.any():
            self.beta_[mask] *= (self.rho / (norms[mask] + 1e-12))[:, None]

    # ---------- public API ----------
    def fit(self, X, y, group=None):
        X = np.asarray(X, dtype=np.float32)
        y = np.asarray(y, dtype=np.float32)
        if group is None:
            group = np.zeros_like(y, dtype=np.int64)

        self._init_params()
        N = X.shape[0]
        eps = 1e-9

        for ep in range(self.epochs):
            self._step += 1
            p, pt, alpha = self._forward(X)
            p_c = np.clip(p, eps, 1 - eps)

            # BCE gradient w.r.t. p
            d_p = -(y / p_c - (1 - y) / (1 - p_c)) / N     # [N]

            # Fairness gradient (only positives)
            pos_mask = y == 1
            if pos_mask.sum() > 1:
                gs = group == 0
                gl = group == 1
                mu_s = p[pos_mask & gs].mean() if (pos_mask & gs).any() else 0.0
                mu_l = p[pos_mask & gl].mean() if (pos_mask & gl).any() else 0.0
                gap = mu_s - mu_l
                # gradient of (gap^2) w.r.t. p_i
                n_s = (pos_mask & gs).sum()
                n_l = (pos_mask & gl).sum()
                d_fair = np.zeros(N)
                if n_s > 0: d_fair[pos_mask & gs] += 2 * gap * (1 / n_s)
                if n_l > 0: d_fair[pos_mask & gl] -= 2 * gap * (1 / n_l)
                d_p = d_p + self.lam_fair * d_fair

            # Backprop into pt via d_p * alpha_t
            d_pt = d_p[:, None] * alpha[None, :]                 # [N, T]

            # Backprop through sigmoid: d_z = d_pt * pt * (1 - pt)
            d_z = d_pt * pt * (1 - pt)                           # [N, T]

            # Grad β_t = X_t^T · d_z_t
            g_beta = np.einsum("nt,ntf->tf", d_z, X)             # [T, F]
            g_beta += self.lam_wd * self.beta_                   # L2 reg
            g_b = d_z.sum(axis=0)                                # [T]

            # Grad γ: through softmax (Jacobian of softmax)
            # d_alpha_t = Σ_i d_p_i * pt_{i,t}
            d_alpha = pt.T @ d_p                                 # [T]
            # softmax jacobian: ∂α_t/∂γ_s = α_t(δ_ts - α_s)
            # d_gamma = α * (d_alpha - (α · d_alpha))
            g_gamma = alpha * (d_alpha - (alpha @ d_alpha))

            # Adam updates
            self.beta_  = self._adam_update("beta",  self.beta_,  g_beta)
            self.b_     = self._adam_update("b",     self.b_,     g_b)
            self.gamma_ = self._adam_update("gamma", self.gamma_, g_gamma)

            # Spectral clipping — Lyapunov stability enforcement
            self._spectral_clip()

        return self

    def predict_proba(self, X):
        X = np.asarray(X, dtype=np.float32)
        p, _, _ = self._forward(X)
        p = np.clip(p, 1e-7, 1 - 1e-7)
        return np.column_stack([1 - p, p])

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)

    def predict_with_mc_dropout(self, X, M=10, dropout_p=None):
        """MC-dropout on gate for uncertainty quantification."""
        if dropout_p is None:
            dropout_p = self.dropout_p
        rng = np.random.default_rng(self.random_state)
        preds = []
        _, pt, alpha = self._forward(X)
        for _ in range(M):
            mask = (rng.random(self.T) > dropout_p).astype(float)
            if mask.sum() == 0:
                mask[rng.integers(0, self.T)] = 1.0
            alpha_drop = alpha * mask
            alpha_drop = alpha_drop / alpha_drop.sum()
            preds.append(pt @ alpha_drop)
        return np.array(preds)  # [M, N]


# ==============================================================================
# RECURRENT BASELINES (LSTM / GRU / RNN) — simplified numpy implementations
# ------------------------------------------------------------------------------
# True PyTorch implementations would be ideal, but for a structural comparison,
# these capture the key recurrent behavior: state propagation + non-linearity.
# For production, swap these for torch.nn.LSTM/GRU/RNN.
# ==============================================================================

class SimpleRNNClassifier(BaseEstimator, ClassifierMixin):
    """
    Minimal recurrent classifier: h_{t+1} = tanh(W_x x_t + W_h h_t + b)
    Final prediction: σ(w^T h_T + c)
    Trained via numeric gradient approximation through an MLP proxy.
    For realistic recurrent behavior we flatten + add recurrence via MLP.
    """
    def __init__(self, kind="rnn", hidden=16, random_state=RANDOM_SEED):
        self.kind = kind
        self.hidden = hidden
        self.random_state = random_state

    def _build_features(self, X):
        """
        Create recurrence-aware features:
          - LSTM: include running mean, trend, volatility per feature
          - GRU:  running mean + last-state bias
          - RNN:  running mean only (weakest recurrence)
        Then feed to an MLP as proxy for sequence model.
        """
        N, T, F = X.shape
        feats = [X.reshape(N, -1)]  # flat

        if self.kind in ("lstm", "gru"):
            # trend (last - first)
            feats.append(X[:, -1, :] - X[:, 0, :])
            # running mean
            feats.append(X.mean(axis=1))
            # volatility (std)
            feats.append(X.std(axis=1))
        if self.kind == "lstm":
            # additional "cell" proxy: exponential moving average
            w = np.exp(np.linspace(-1, 0, T))
            w = w / w.sum()
            feats.append((X * w[None, :, None]).sum(axis=1))
        if self.kind == "rnn":
            feats.append(X.mean(axis=1))
        return np.concatenate(feats, axis=1)

    def fit(self, X, y):
        X2 = self._build_features(X)
        # LSTM is larger/stronger; RNN is smaller & more unstable
        if self.kind == "lstm":
            self.model_ = MLPClassifier(hidden_layer_sizes=(self.hidden, self.hidden),
                                        activation="tanh",
                                        max_iter=600, random_state=self.random_state,
                                        alpha=1e-4)
        elif self.kind == "gru":
            self.model_ = MLPClassifier(hidden_layer_sizes=(self.hidden,),
                                        activation="tanh",
                                        max_iter=500, random_state=self.random_state,
                                        alpha=1e-4)
        else:  # rnn
            self.model_ = MLPClassifier(hidden_layer_sizes=(self.hidden,),
                                        activation="tanh",
                                        max_iter=300, random_state=self.random_state,
                                        alpha=1e-5)
        self.model_.fit(X2, y)
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(self._build_features(X))

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)


# ==============================================================================
# MODEL REGISTRY — wraps all models with uniform fit/predict interface
# ==============================================================================

def flatten_X(X):
    """[N, T, F] → [N, T*F]"""
    return X.reshape(X.shape[0], -1)


class FlatWrapper(BaseEstimator, ClassifierMixin):
    """Wraps a tabular model to accept 3D [N,T,F] input by flattening."""
    def __init__(self, base):
        self.base = base

    def fit(self, X, y, **kwargs):
        self.model_ = clone(self.base)
        self.model_.fit(flatten_X(X), y)
        return self

    def predict_proba(self, X):
        return self.model_.predict_proba(flatten_X(X))

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X)[:, 1] >= threshold).astype(int)


def build_all_models() -> dict:
    """All competitors + AHNN. Keys are display names."""
    models = {
        "AHNN":       AHNNModel(),
        "LSTM":       SimpleRNNClassifier(kind="lstm"),
        "GRU":        SimpleRNNClassifier(kind="gru"),
        "RNN":        SimpleRNNClassifier(kind="rnn"),
        "RandomForest": FlatWrapper(RandomForestClassifier(
                            n_estimators=100, max_depth=5,
                            random_state=RANDOM_SEED, n_jobs=-1)),
        "GBM":        FlatWrapper(GradientBoostingClassifier(
                            n_estimators=100, max_depth=3,
                            learning_rate=0.1, random_state=RANDOM_SEED)),
        "SVM-RBF":    FlatWrapper(SVC(kernel="rbf", C=1.0, probability=True,
                                      random_state=RANDOM_SEED)),
        "MLP":        FlatWrapper(MLPClassifier(hidden_layer_sizes=(32, 16),
                                                max_iter=500,
                                                random_state=RANDOM_SEED)),
        "LogReg":     FlatWrapper(LogisticRegression(C=1.0, max_iter=2000,
                                                     random_state=RANDOM_SEED)),
    }
    if HAS_XGB:
        models["XGBoost"] = FlatWrapper(XGBClassifier(
            n_estimators=100, max_depth=4, learning_rate=0.1,
            use_label_encoder=False, eval_metric="logloss",
            random_state=RANDOM_SEED, verbosity=0))
    if HAS_LGBM:
        models["LightGBM"] = FlatWrapper(LGBMClassifier(
            n_estimators=100, max_depth=4, learning_rate=0.1,
            random_state=RANDOM_SEED, verbosity=-1))
    return models


# ==============================================================================
# METRIC COMPUTATIONS
# ==============================================================================

def hamming_distance(y_true, y_pred):
    """d_H = misclassification rate = 1 - accuracy (PDF §6.1)."""
    return float(np.mean(y_true != y_pred))


def euclidean_prob_distance(y_true, y_prob):
    """d_E = √((1/n) Σ (p-y)²)  (PDF §6.2 — √Brier)."""
    return float(np.sqrt(np.mean((y_prob - y_true) ** 2)))


def lyapunov_exponent(model, X, epsilon=0.01, n_perturb=20):
    """
    Approximate: λ ≈ log(||Δp|| / ||Δx||)
    PDF says AHNN ≈ -0.2 (near zero, stable), LightGBM ≈ -11.9 (over-contractive).
    """
    X = np.asarray(X, dtype=np.float32)
    rng = np.random.default_rng(RANDOM_SEED)
    ratios = []
    try:
        p_orig = model.predict_proba(X)[:, 1]
    except Exception:
        return np.nan
    for _ in range(n_perturb):
        delta = epsilon * rng.standard_normal(X.shape).astype(np.float32)
        Xp = X + delta
        try:
            p_new = model.predict_proba(Xp)[:, 1]
        except Exception:
            continue
        dx_norm = np.linalg.norm(delta)
        dp_norm = np.linalg.norm(p_new - p_orig)
        if dx_norm > 0 and dp_norm > 1e-12:
            ratios.append(np.log(dp_norm / dx_norm))
    if not ratios:
        return np.nan
    return float(np.mean(ratios))


def fairness_gap(y_true, y_prob, group):
    """
    |TPR_small - TPR_large|  using probability means on positives (PDF §5.1).
    """
    y_true = np.asarray(y_true)
    group  = np.asarray(group)
    pos = y_true == 1
    if pos.sum() < 2:
        return np.nan
    gs = (group == 0) & pos
    gl = (group == 1) & pos
    if gs.sum() == 0 or gl.sum() == 0:
        return np.nan
    return float(abs(y_prob[gs].mean() - y_prob[gl].mean()))


def pred_volatility(model_factory, X_tr, y_tr, X_te, n_boot=5):
    """
    Bootstrap-based predictive volatility = mean std of p across bootstraps.
    Lower is better.
    """
    rng = np.random.default_rng(RANDOM_SEED)
    N_tr = len(y_tr)
    preds = []
    for b in range(n_boot):
        idx = rng.integers(0, N_tr, size=N_tr)
        if len(np.unique(y_tr[idx])) < 2:
            continue
        try:
            m = model_factory()
            m.fit(X_tr[idx], y_tr[idx])
            preds.append(m.predict_proba(X_te)[:, 1])
        except Exception:
            continue
    if len(preds) < 2:
        return np.nan
    return float(np.mean(np.std(np.array(preds), axis=0)))


# ==============================================================================
# EVALUATION LOOP — k-fold over all models
# ==============================================================================

def evaluate_all_models(ds: AHNNDataset):
    """
    Returns a dict of per-model metrics (averaged across folds), plus
    per-fold AUC matrix for Friedman test.
    """
    models = build_all_models()
    metric_names = ["Accuracy", "F1", "AUC", "PR-AUC", "Brier",
                    "Hamming", "Euclidean", "Lyapunov",
                    "FairnessGap", "PredVol"]
    # Per-fold storage
    per_fold_auc = {m: [] for m in models}
    per_fold_f1  = {m: [] for m in models}
    agg_metrics  = {m: {k: [] for k in metric_names} for m in models}

    for fold_idx, (tr_idx, te_idx) in enumerate(ds.kfold_splits):
        print(f"   [Fold {fold_idx+1}/{len(ds.kfold_splits)}] training all models ...")
        X_tr, X_te = ds.X[tr_idx], ds.X[te_idx]
        y_tr, y_te = ds.y[tr_idx], ds.y[te_idx]
        g_tr, g_te = ds.group[tr_idx], ds.group[te_idx]

        # Standardize per fold
        X_tr_s, X_te_s, _ = standardize_tensor(X_tr, X_te)

        if len(np.unique(y_te)) < 2 or len(np.unique(y_tr)) < 2:
            continue

        for name, base in models.items():
            try:
                # AHNN uses `group` in fit; others don't
                m = clone(base) if hasattr(base, "get_params") else base
                if name == "AHNN":
                    m = AHNNModel()    # fresh
                    m.fit(X_tr_s, y_tr, group=g_tr)
                else:
                    m.fit(X_tr_s, y_tr)

                proba = m.predict_proba(X_te_s)[:, 1]
                pred  = (proba >= 0.5).astype(int)

                acc  = accuracy_score(y_te, pred)
                f1   = f1_score(y_te, pred, zero_division=0)
                auc  = roc_auc_score(y_te, proba) if len(np.unique(y_te)) == 2 else np.nan
                pr   = average_precision_score(y_te, proba)
                br   = brier_score_loss(y_te, proba)
                ham  = hamming_distance(y_te, pred)
                eu   = euclidean_prob_distance(y_te, proba)
                lyap = lyapunov_exponent(m, X_te_s)
                fgap = fairness_gap(y_te, proba, g_te)

                # Predictive volatility (bootstrap) — use factory
                def factory(_name=name, _base=base):
                    if _name == "AHNN":
                        return AHNNModel()
                    return clone(_base)
                pv = pred_volatility(factory, X_tr_s, y_tr, X_te_s, n_boot=5)

                per_fold_auc[name].append(auc if not np.isnan(auc) else 0.5)
                per_fold_f1[name].append(f1)

                agg_metrics[name]["Accuracy"].append(acc)
                agg_metrics[name]["F1"].append(f1)
                agg_metrics[name]["AUC"].append(auc)
                agg_metrics[name]["PR-AUC"].append(pr)
                agg_metrics[name]["Brier"].append(br)
                agg_metrics[name]["Hamming"].append(ham)
                agg_metrics[name]["Euclidean"].append(eu)
                agg_metrics[name]["Lyapunov"].append(lyap)
                agg_metrics[name]["FairnessGap"].append(fgap)
                agg_metrics[name]["PredVol"].append(pv)

            except Exception as e:
                print(f"     [WARN] {name} failed: {e}")
                for k in metric_names:
                    agg_metrics[name][k].append(np.nan)

    # Average per model
    summary = {}
    for name in models:
        row = {}
        for k in metric_names:
            arr = np.array(agg_metrics[name][k], dtype=float)
            row[k] = float(np.nanmean(arr)) if arr.size and not np.all(np.isnan(arr)) else np.nan
        summary[name] = row

    df = pd.DataFrame(summary).T[metric_names]
    return {
        "summary": df,
        "per_fold_auc": per_fold_auc,
        "per_fold_f1":  per_fold_f1,
        "model_names":  list(models.keys()),
    }


# ==============================================================================
# COMPOSITE RANKING (PDF §10)
# ==============================================================================

def normalize_metrics(df: pd.DataFrame):
    """
    Benefit (↑): Accuracy, F1, AUC, PR-AUC
    Cost (↓):   Brier, Hamming, Euclidean, FairnessGap, PredVol
    Lyapunov: special — closer to 0 is better (AHNN), so use |·| and invert
    """
    benefit = ["Accuracy", "F1", "AUC", "PR-AUC"]
    cost    = ["Brier", "Hamming", "Euclidean", "FairnessGap", "PredVol"]

    Z = pd.DataFrame(index=df.index)
    for col in benefit:
        x = df[col].values.astype(float)
        lo, hi = np.nanmin(x), np.nanmax(x)
        Z[col] = (x - lo) / (hi - lo + 1e-12)
    for col in cost:
        x = df[col].values.astype(float)
        lo, hi = np.nanmin(x), np.nanmax(x)
        Z[col] = 1 - (x - lo) / (hi - lo + 1e-12)
    # Lyapunov → distance from 0 (lower |λ| better = closer to ideal stability)
    lyap = np.abs(df["Lyapunov"].values.astype(float))
    lo, hi = np.nanmin(lyap), np.nanmax(lyap)
    Z["Lyapunov"] = 1 - (lyap - lo) / (hi - lo + 1e-12)

    return Z.fillna(0)


def composite_scores(df: pd.DataFrame):
    Z = normalize_metrics(df)

    # C_equal: simple mean over all normalized metrics
    Z_equal = Z.mean(axis=1)

    # C_regulator: 0.4 Accuracy + 0.3 Stability + 0.3 Fairness (PDF §10.2)
    acc_group    = Z[["Accuracy", "F1", "AUC", "PR-AUC", "Brier"]].mean(axis=1)
    stab_group   = Z[["PredVol", "Euclidean", "Lyapunov"]].mean(axis=1)
    fair_group   = Z[["FairnessGap", "Hamming"]].mean(axis=1)
    Z_reg = 0.40 * acc_group + 0.30 * stab_group + 0.30 * fair_group

    out = pd.DataFrame({
        "C_equal":     Z_equal,
        "C_regulator": Z_reg,
        "AccGroup":    acc_group,
        "StabGroup":   stab_group,
        "FairGroup":   fair_group,
    })
    return out.sort_values("C_regulator", ascending=False)


# ==============================================================================
# PLOTS
# ==============================================================================

def plot_model_comparison_multimetric(summary_df: pd.DataFrame, out_name="11_ahnn_vs_all.png"):
    """
    Multi-panel: per-metric bar comparison with AHNN highlighted.
    """
    metrics_higher_better = ["Accuracy", "F1", "AUC", "PR-AUC"]
    metrics_lower_better  = ["Brier", "Hamming", "Euclidean", "FairnessGap", "PredVol"]

    all_metrics = metrics_higher_better + metrics_lower_better
    n_panels = len(all_metrics)
    cols = 3
    rows = int(np.ceil(n_panels / cols))

    fig, axes = plt.subplots(rows, cols, figsize=(16, 3.2 * rows))
    axes = axes.ravel()

    for i, metric in enumerate(all_metrics):
        ax = axes[i]
        s = summary_df[metric].copy()
        if metric in metrics_higher_better:
            s = s.sort_values(ascending=False)
            is_higher = True
        else:
            s = s.sort_values(ascending=True)
            is_higher = False

        colors = ["#d62728" if name == "AHNN" else "#1f77b4" for name in s.index]
        bars = ax.barh(s.index[::-1], s.values[::-1],
                       color=colors[::-1], edgecolor="black", alpha=0.85)
        ax.set_title(f"{metric} ({'↑ higher better' if is_higher else '↓ lower better'})",
                     fontweight="bold")
        for bar, val in zip(bars, s.values[::-1]):
            if not np.isnan(val):
                ax.text(val + (0.003 if is_higher else 0.003),
                        bar.get_y() + bar.get_height() / 2,
                        f"{val:.3f}", va="center", fontsize=8)
        ax.grid(alpha=0.3, axis="x")

    # Hide unused axes
    for j in range(n_panels, len(axes)):
        axes[j].axis("off")

    fig.suptitle("Figure 11: AHNN vs All Competitors — Multi-Metric Comparison",
                 fontsize=14, fontweight="bold", y=1.00)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{out_name}", bbox_inches="tight")
    plt.close()


def plot_composite_ranking(comp_df: pd.DataFrame, out_name="12_composite_ranking.png"):
    """Bar chart of Equal + Regulator-aligned composite (PDF §10)."""
    fig, axes = plt.subplots(1, 2, figsize=(15, 6))

    for ax, col, title in [
        (axes[0], "C_equal",     "Equal-Weight Composite (Cₑqᵤₐₗ)"),
        (axes[1], "C_regulator", "Regulator-Aligned Composite (Cᵣₑ𝓰)\n0.4·Acc + 0.3·Stab + 0.3·Fair"),
    ]:
        s = comp_df[col].sort_values(ascending=True)
        colors = ["#d62728" if name == "AHNN" else "#1f77b4" for name in s.index]
        bars = ax.barh(s.index, s.values, color=colors, edgecolor="black", alpha=0.85)
        for bar, val in zip(bars, s.values):
            ax.text(val + 0.005, bar.get_y() + bar.get_height() / 2,
                    f"{val:.3f}", va="center", fontsize=9, fontweight="bold")
        ax.set_xlabel("Composite score (higher = better)")
        ax.set_title(title, fontweight="bold")
        ax.grid(alpha=0.3, axis="x")
        ax.set_xlim(0, 1.08)

    fig.suptitle("Figure 12: Composite Ranking — AHNN vs Competitors (Basel III aligned)",
                 fontsize=13, fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{out_name}", bbox_inches="tight")
    plt.close()


def plot_friedman_test(per_fold_auc: dict, out_name="13_friedman_test.png"):
    """Friedman test on per-fold AUCs (AHNN vs all)."""
    names = [n for n, v in per_fold_auc.items() if len(v) >= 2]
    n_folds_per = [len(per_fold_auc[n]) for n in names]
    max_f = max(n_folds_per)
    # keep models with full fold count
    good = [n for n in names if len(per_fold_auc[n]) == max_f]
    mat = np.array([per_fold_auc[n] for n in good]).T   # [folds, models]

    stat, pval = stats.friedmanchisquare(*[mat[:, j] for j in range(mat.shape[1])])

    ranks = np.zeros_like(mat)
    for i in range(mat.shape[0]):
        ranks[i] = stats.rankdata(-mat[i])
    avg_ranks = ranks.mean(axis=0)
    mean_auc  = mat.mean(axis=0)
    std_auc   = mat.std(axis=0)

    # Sort by mean AUC (best first)
    order = np.argsort(-mean_auc)
    good_s   = [good[i] for i in order]
    mat_s    = mat[:, order]
    mean_s   = mean_auc[order]
    std_s    = std_auc[order]
    ranks_s  = avg_ranks[order]
    colors   = ["#d62728" if n == "AHNN" else "#1f77b4" for n in good_s]

    fig = plt.figure(figsize=(16, 9))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.2, 1], height_ratios=[1, 1],
                          hspace=0.45, wspace=0.28)

    # TOP-LEFT: per-fold lines
    ax1 = fig.add_subplot(gs[0, 0])
    cmap = plt.cm.tab20
    for j, nm in enumerate(good_s):
        c = "red" if nm == "AHNN" else cmap(j % 20)
        lw = 2.5 if nm == "AHNN" else 1.5
        ax1.plot(range(1, mat.shape[0] + 1), mat_s[:, j],
                 marker="o", linewidth=lw, color=c,
                 label=f"{nm} ({mean_s[j]:.3f})",
                 alpha=1.0 if nm == "AHNN" else 0.75)
    ax1.set_xlabel("Fold"); ax1.set_ylabel("Test AUC")
    ax1.set_title("Per-fold AUC — AHNN vs all")
    ax1.set_xticks(range(1, mat.shape[0] + 1))
    ax1.legend(fontsize=7, loc="lower right", ncol=2)
    ax1.grid(alpha=0.3)

    # TOP-RIGHT: boxplot
    ax2 = fig.add_subplot(gs[0, 1])
    bp = ax2.boxplot([mat_s[:, j] for j in range(len(good_s))],
                     tick_labels=good_s, patch_artist=True, widths=0.55)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color); patch.set_alpha(0.55)
    ax2.set_ylabel("AUC"); ax2.set_title("AUC distribution across folds")
    ax2.tick_params(axis="x", rotation=40)
    ax2.grid(alpha=0.3, axis="y")

    # BOTTOM-LEFT: mean AUC bar with error
    ax3 = fig.add_subplot(gs[1, 0])
    xpos = np.arange(len(good_s))
    ax3.bar(xpos, mean_s, yerr=std_s, capsize=4,
            color=colors, edgecolor="black", alpha=0.85)
    for x, v in zip(xpos, mean_s):
        ax3.text(x, v + 0.008, f"{v:.3f}", ha="center",
                 fontsize=8, fontweight="bold")
    ax3.set_xticks(xpos)
    ax3.set_xticklabels(good_s, rotation=40, ha="right")
    ax3.set_ylabel("Mean AUC ± std")
    ax3.set_title("Mean Test AUC")
    ax3.set_ylim(max(0, mean_s.min() - 0.08), min(1.05, mean_s.max() + 0.05))
    ax3.grid(alpha=0.3, axis="y")

    # BOTTOM-RIGHT: avg ranks
    ax4 = fig.add_subplot(gs[1, 1])
    rord = np.argsort(ranks_s)
    names_rord = [good_s[i] for i in rord[::-1]]
    vals_rord  = [ranks_s[i] for i in rord[::-1]]
    cols_rord  = [colors[i] for i in rord[::-1]]
    ax4.barh(names_rord, vals_rord, color=cols_rord,
             edgecolor="black", alpha=0.85)
    ax4.set_xlabel("Avg Rank (lower = better)")
    ax4.set_title("Friedman Ranks")
    ax4.grid(alpha=0.3, axis="x")

    verdict = "REJECT H0 — models differ" if pval < 0.05 else "FAIL to reject H0"
    fig.text(0.5, -0.02,
             f"Friedman χ² = {stat:.3f}   |   df = {len(good_s) - 1}   |   "
             f"p-value = {pval:.4f}   →   {verdict}",
             ha="center", fontsize=11, fontweight="bold",
             bbox=dict(boxstyle="round,pad=0.5",
                       facecolor="lightyellow", edgecolor="black"))
    fig.suptitle(f"Figure 13: Friedman Test — AHNN vs {len(good_s)-1} Competitors "
                 f"across {mat.shape[0]} Folds",
                 fontsize=14, fontweight="bold", y=1.00)
    plt.tight_layout()
    plt.savefig(f"{OUT_DIR}/{out_name}", bbox_inches="tight")
    plt.close()


def plot_ks_test(ds: AHNNDataset, out_name="14_ks_test.png"):
    X_company_mean = ds.X.mean(axis=1)
    y = ds.y
    rows = []
    for j, f in enumerate(FEATURE_NAMES):
        vs, vd = X_company_mean[y == 0, j], X_company_mean[y == 1, j]
        if len(vs) < 2 or len(vd) < 2:
            rows.append({"feature": f, "ks": np.nan, "p": np.nan})
            continue
        s, p = stats.ks_2samp(vs, vd)
        rows.append({"feature": f, "ks": s, "p": p, "vs": vs, "vd": vd})
    rows.sort(key=lambda r: -r["ks"] if not np.isnan(r["ks"]) else 0)

    fig = plt.figure(figsize=(14, 8))
    gs = fig.add_gridspec(2, 2, width_ratios=[1, 1.1], hspace=0.42, wspace=0.28)

    ax1 = fig.add_subplot(gs[0, 0])
    names = [r["feature"] for r in rows]
    kvals = [r["ks"] for r in rows]
    pvals = [r["p"] for r in rows]
    colors = ["#d62728" if p < 0.05 else "#7f7f7f" for p in pvals]
    ax1.barh(names[::-1], kvals[::-1], color=colors[::-1], edgecolor="black")
    ax1.axvline(0.3, color="green", linestyle="--", label="KS=0.30 acceptable")
    ax1.axvline(0.5, color="orange", linestyle="--", label="KS=0.50 excellent")
    ax1.set_xlabel("KS Statistic"); ax1.set_title("KS Statistic per Feature")
    ax1.legend(fontsize=8, loc="lower right"); ax1.grid(alpha=0.3, axis="x")

    ax2 = fig.add_subplot(gs[0, 1])
    neg_log_p = [-np.log10(max(p, 1e-10)) for p in pvals]
    ax2.barh(names[::-1], neg_log_p[::-1], color=colors[::-1], edgecolor="black")
    ax2.axvline(-np.log10(0.05), color="red", linestyle="--", label="p=0.05")
    ax2.set_xlabel("-log10(p)"); ax2.set_title("Statistical Significance")
    ax2.legend(fontsize=8); ax2.grid(alpha=0.3, axis="x")

    ax3 = fig.add_subplot(gs[1, :])
    top3 = [r for r in rows if "vs" in r][:3]
    cols = ["#1f77b4", "#ff7f0e", "#2ca02c"]
    for k, r in enumerate(top3):
        s_sorted = np.sort(r["vs"]); d_sorted = np.sort(r["vd"])
        s_cdf = np.arange(1, len(s_sorted) + 1) / len(s_sorted)
        d_cdf = np.arange(1, len(d_sorted) + 1) / len(d_sorted)
        ax3.plot(s_sorted, s_cdf, color=cols[k], linewidth=2,
                 label=f"{r['feature']} | Safe (KS={r['ks']:.3f})")
        ax3.plot(d_sorted, d_cdf, color=cols[k], linewidth=2, linestyle="--",
                 label=f"{r['feature']} | Default")
    ax3.set_xlabel("Feature value"); ax3.set_ylabel("Empirical CDF")
    ax3.set_title("CDF Comparison — Top-3 Discriminative Features")
    ax3.legend(fontsize=8, ncol=3, loc="lower right"); ax3.grid(alpha=0.3)

    fig.suptitle("Figure 14: Kolmogorov–Smirnov Test — Feature Discriminative Power",
                 fontsize=13, fontweight="bold", y=1.00)
    plt.savefig(f"{OUT_DIR}/{out_name}", bbox_inches="tight")
    plt.close()


def plot_prediction_variance_ahnn(ds: AHNNDataset,
                                  n_boot: int = ENSEMBLE_MEMBERS,
                                  out_name="15_prediction_variance.png"):
    """Bootstrap prediction variance using AHNN (thesis model)."""
    tr_idx, te_idx = ds.kfold_splits[0]
    X_tr, X_te = ds.X[tr_idx], ds.X[te_idx]
    y_tr, y_te = ds.y[tr_idx], ds.y[te_idx]
    g_tr       = ds.group[tr_idx]
    X_tr_s, X_te_s, _ = standardize_tensor(X_tr, X_te)

    rng = np.random.default_rng(RANDOM_SEED)
    n_tr = len(tr_idx); n_te = len(te_idx)
    all_preds = np.zeros((n_boot, n_te))
    for b in range(n_boot):
        idx = rng.integers(0, n_tr, size=n_tr)
        if len(np.unique(y_tr[idx])) < 2:
            all_preds[b] = 0.5; continue
        try:
            m = AHNNModel(random_state=RANDOM_SEED + b)
            m.fit(X_tr_s[idx], y_tr[idx], group=g_tr[idx])
            all_preds[b] = m.predict_proba(X_te_s)[:, 1]
        except Exception as e:
            print(f"   [WARN] bootstrap {b} failed: {e}")
            all_preds[b] = 0.5

    pred_mean = all_preds.mean(axis=0)
    pred_std  = all_preds.std(axis=0)
    order = np.argsort(pred_mean)
    pm, ps, ys = pred_mean[order], pred_std[order], y_te[order]
    ps_sorted = all_preds[:, order]

    fig = plt.figure(figsize=(14, 8))
    gs = fig.add_gridspec(2, 2, width_ratios=[1.4, 1], height_ratios=[1.2, 1],
                          hspace=0.42, wspace=0.28)

    ax1 = fig.add_subplot(gs[0, :])
    x = np.arange(n_te)
    ax1.fill_between(x, np.clip(pm - ps, 0, 1), np.clip(pm + ps, 0, 1),
                     color="#d62728", alpha=0.25, label="±1 std")
    ax1.plot(x, pm, color="#d62728", linewidth=1.8, label="Mean P(default)")
    for b in range(n_boot):
        ax1.scatter(x, ps_sorted[b], s=8, alpha=0.3, color="gray")
    smask = ys == 0; dmask = ys == 1
    ax1.scatter(x[smask], pm[smask], color="#2ca02c", s=50, edgecolor="black",
                linewidth=0.7, label="True: Safe", zorder=5)
    ax1.scatter(x[dmask], pm[dmask], color="#d62728", s=50, edgecolor="black",
                linewidth=0.7, label="True: Default", zorder=5)
    ax1.axhline(0.5, color="black", linestyle=":", alpha=0.6)
    ax1.set_xlabel("Test company (sorted by mean prediction)")
    ax1.set_ylabel("Predicted P(default)")
    ax1.set_title(f"AHNN Prediction mean & variance across {n_boot} bootstraps")
    ax1.legend(fontsize=9, loc="upper left", ncol=2); ax1.set_ylim(-0.05, 1.05)
    ax1.grid(alpha=0.3)

    ax2 = fig.add_subplot(gs[1, 0])
    ax2.hist(pred_std, bins=15, color="#ff7f0e", edgecolor="black", alpha=0.85)
    ax2.axvline(pred_std.mean(), color="red", linestyle="--", linewidth=2,
                label=f"Mean std = {pred_std.mean():.3f}")
    ax2.set_xlabel("Std per company"); ax2.set_ylabel("Count")
    ax2.set_title("Distribution of predictive volatility"); ax2.legend()
    ax2.grid(alpha=0.3)

    ax3 = fig.add_subplot(gs[1, 1])
    for yv, c, lbl in [(0, "#2ca02c", "Safe"), (1, "#d62728", "Default")]:
        mk = y_te == yv
        ax3.scatter(pred_mean[mk], pred_std[mk], color=c, s=60,
                    edgecolor="black", linewidth=0.7, alpha=0.8, label=lbl)
    ax3.set_xlabel("Mean P(default)"); ax3.set_ylabel("Std")
    ax3.set_title("Stability profile"); ax3.legend(); ax3.grid(alpha=0.3)

    fig.suptitle(f"Figure 15: AHNN Prediction Variance across {n_boot} Bootstrap Samples",
                 fontsize=13, fontweight="bold", y=1.00)
    plt.savefig(f"{OUT_DIR}/{out_name}", bbox_inches="tight")
    plt.close()


# ==============================================================================
# MAIN
# ==============================================================================

def main():
    print(">> Step 1: Building AHNN dataset (synthetic) ...")
    raw = generate_synthetic_raw()
    ds = build_dataset_from_raw(raw)
    print(ds.summary())

    print("\n>> Step 2: Training & evaluating ALL models across 5 folds ...")
    print("   (this takes a minute — AHNN + 10 competitors × 5 folds)")
    results = evaluate_all_models(ds)
    summary_df = results["summary"]

    print("\n>> Per-model metrics (averaged across folds):")
    pd.set_option("display.float_format", lambda x: f"{x:.4f}")
    print(summary_df.to_string())

    # Composite
    comp_df = composite_scores(summary_df)
    print("\n>> Composite Ranking (Basel III aligned):")
    print(comp_df[["C_equal", "C_regulator"]].to_string())

    # -------------------- Figures --------------------
    print("\n>> Generating all 5 comparison figures ...")
    plot_model_comparison_multimetric(summary_df)
    print("   [Saved] 11_ahnn_vs_all.png")

    plot_composite_ranking(comp_df)
    print("   [Saved] 12_composite_ranking.png")

    plot_friedman_test(results["per_fold_auc"])
    print("   [Saved] 13_friedman_test.png")

    plot_ks_test(ds)
    print("   [Saved] 14_ks_test.png")

    plot_prediction_variance_ahnn(ds, n_boot=ENSEMBLE_MEMBERS)
    print("   [Saved] 15_prediction_variance.png")

    # Save metrics CSV for thesis appendix
    summary_df.to_csv(f"{OUT_DIR}/all_models_metrics.csv", float_format="%.4f")
    comp_df.to_csv(f"{OUT_DIR}/composite_ranking.csv", float_format="%.4f")
    print(f"\n   [Saved] all_models_metrics.csv, composite_ranking.csv")

    # Final verdict
    best_eq  = comp_df["C_equal"].idxmax()
    best_reg = comp_df["C_regulator"].idxmax()
    print(f"\n>> FINAL: Best by Equal-Weight Composite    = {best_eq}")
    print(f"          Best by Regulator-Aligned Composite = {best_reg}")
    print(f"\n>> All outputs saved to: {os.path.abspath(OUT_DIR)}")
    print(">> Done.")


if __name__ == "__main__":
    main()

>> Step 1: Building AHNN dataset (synthetic) ...
============ AHNNDataset Summary ============
Train: 49 samples, 5 features
Test:  13 samples, 5 features
Features: ['feature_0', 'feature_1', 'feature_2', 'feature_3', 'feature_4', 'feature_5', 'feature_6', 'feature_7', 'feature_8', 'feature_9']
T_years: 5, F_features: 10, N_companies: 62

>> Step 2: Training & evaluating ALL models across 5 folds ...
   (this takes a minute — AHNN + 10 competitors × 5 folds)
   [Fold 1/5] training all models ...


TypeError: standardize_tensor() takes 1 positional argument but 2 were given